# Seleção de variáveis em cenários bancários sintéticos: com Boruta

Este notebook compara Boruta com cinco estimadores: **Bankai Random Forest instalado pelo PyPI**, Random Forest do scikit-learn, Extra Trees, LightGBM RF e XGBoost RF. O notebook irmão `banking_feature_importance_benchmark.ipynb` mede a importância nativa dos mesmos estimadores sem Boruta, usando a mesma geração de dados e seeds.

Cada cenário tem features relevantes conhecidas por construção. A avaliação mede recall das relevantes, FDR, número selecionado, rodada em que a qualidade alvo é atingida e runtime do Boruta. Isto avalia seleção de variáveis, não desempenho preditivo fora da amostra.

`LIGHT_MODE = True` reduz dimensões e repetições para desenvolvimento local. Defina `False` para executar todos os dados: 500.000 linhas, 500 features e 10 medições por modelo/cenário. Cada par cenário/modelo tem um aquecimento. O Bankai é instalado do PyPI e a origem do módulo é validada.

Boruta e a medição sem Boruta usam a mesma partição de treino de 80% para cada cenário/repetição; as mesmas seeds reproduzem dados e partição. O holdout de 20% fica separado da seleção de features.

## 1. Instalar dependências

In [ ]:
%pip install --upgrade --force-reinstall bankai-random-forest==0.1.0a10 scikit-learn==1.9.1 boruta==0.4.3 lightgbm==4.7.0 xgboost==3.2.0 psutil==7.2.2 threadpoolctl==3.7.2 pandas==3.0.6 numpy==2.4.6 scipy==1.17.1

Reinicie o kernel depois da instalação, se o Jupyter solicitar. A instalação é do índice configurado pelo pip (PyPI por padrão).

## 2. Recursos disponíveis nesta máquina

In [ ]:
import os, platform, sys
import psutil
from threadpoolctl import threadpool_info

CPU_COUNT = os.cpu_count() or 1
physical = psutil.cpu_count(logical=False) or CPU_COUNT
memory = psutil.virtual_memory()
print({
    'machine': platform.machine(), 'platform': platform.platform(),
    'python': sys.version, 'logical_cpus': CPU_COUNT,
    'physical_cores': physical, 'memory_total_gib': round(memory.total / 2**30, 2),
    'memory_available_gib': round(memory.available / 2**30, 2),
    'threadpools_before_import': threadpool_info(),
})
N_JOBS = CPU_COUNT  # todos os núcleos lógicos disponíveis

## 3. Configuração

In [ ]:
from pathlib import Path

LIGHT_MODE = True  # mude para False para o benchmark completo
BASE_SEED = 20261011
N_WARMUP = 1
TARGET_RECALL = 0.95
MAX_FDR = 0.05
WORKDIR = Path.cwd().resolve()
REPO = WORKDIR.parent if WORKDIR.name == 'benchmarks' else WORKDIR
RESULT_DIR = REPO / 'benchmarks' / 'results-boruta-banking-synthetic'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = RESULT_DIR / ('raw-light.csv' if LIGHT_MODE else 'raw-full.csv')

if LIGHT_MODE:
    N_ROWS, N_FEATURES, N_INFORMATIVE = 20_000, 100, 24
    N_REPEATS, N_TREES, MAX_ROUNDS = 1, 20, 8
else:
    N_ROWS, N_FEATURES, N_INFORMATIVE = 500_000, 500, 24
    N_REPEATS, N_TREES, MAX_ROUNDS = 10, 100, 30

MAX_BINS = 63
ROW_FRACTION = 0.80
print({'light_mode': LIGHT_MODE, 'rows_per_scenario': N_ROWS,
       'features': N_FEATURES, 'informative_features': N_INFORMATIVE,
       'measured_repeats_per_model_scenario': N_REPEATS,
       'warmups_per_model_scenario': N_WARMUP, 'trees': N_TREES,
       'boruta_max_rounds': MAX_ROUNDS, 'n_jobs': N_JOBS,
       'result_file': str(RAW_PATH)})

### Recursos e duração

O modo completo tem 4 cenários × 5 estimadores × 10 medições, além de um aquecimento para cada par cenário/estimador: **220 ajustes Boruta**. Boruta duplica temporariamente a matriz para criar features sombra. Com 500.000 × 500 valores float32, a matriz original ocupa cerca de 0,93 GiB; a matriz aumentada passa de 1,8 GiB, sem contar árvores e cópias internas. Verifique a memória disponível antes de desligar o modo light.

Todos os estimadores recebem todos os CPUs lógicos detectados. As rodadas Boruta são sequenciais; cada ajuste de floresta pode usar todos os núcleos.

## 4. Os quatro cenários

1. **Inadimplência de crédito (1,5% positivos):** prever atraso grave futuro; sinais de comprometimento de renda, uso de crédito, histórico, emprego e idade de crédito. Sinais fracos e correlações motivam árvores moderadas e folhas mínimas maiores.
2. **Fraude em transações (0,2% positivos):** valor atípico, horário/dispositivo incomum, velocidade e interações. Evento muito raro motiva árvores um pouco mais profundas para capturar combinações, mantendo folhas mínimas para limitar sobreajuste.
3. **Churn (6% positivos):** queda de atividade e saldo, reclamações, tarifas e recência de contato. Relações por segmento justificam profundidade intermediária.
4. **Aceitação de oferta (18% positivos):** capacidade financeira, engajamento, adequação e canal. Mais positivos e sinais relativamente diretos permitem árvores mais rasas.

Cada cenário tem 24 features informativas nos índices 0–23 e ruído nas restantes. Há proxies correlacionadas, limiares, curvas e interações. Nenhuma coluna revela o rótulo diretamente. São hipóteses sintéticas controladas, não retratos de clientes reais.

## 5. Importações e confirmação do Bankai vindo do PyPI

In [ ]:
import gc, json, math, os, time, warnings
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
from boruta import BorutaPy
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from threadpoolctl import threadpool_limits
import bankai_random_forest
from bankai_random_forest import BankaiRandomForestClassifier

repo = REPO
package_file = Path(bankai_random_forest.__file__).resolve()
extension_file = Path(bankai_random_forest._core.__file__).resolve()
dist = metadata.distribution('bankai-random-forest')
direct_url = dist.read_text('direct_url.json')
if package_file.is_relative_to(repo) or extension_file.is_relative_to(repo):
    raise RuntimeError(f'Bankai foi importado do checkout local, esperado PyPI: {package_file}')
if direct_url and json.loads(direct_url).get('dir_info', {}).get('editable', False):
    raise RuntimeError('A instalação do Bankai está em modo editable; reinstale pelo PyPI e reinicie o kernel.')
def installed_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        # Some notebook environments expose an importable module without its
        # wheel metadata. Keep the environment report useful in that case.
        return 'metadata unavailable'

versions = {
    'bankai': dist.version,
    'bankai_package': str(package_file),
    'bankai_native_extension': str(extension_file),
    'scikit_learn': installed_version('scikit-learn'),
    'lightgbm': installed_version('lightgbm'),
    'xgboost': installed_version('xgboost'),
}
if 'boruta' in globals():
    versions['boruta'] = installed_version('Boruta')
print(versions)

## 6. Geradores dos datasets

As features 0–23 são informativas por construção; a tabela `SCENARIOS` registra prevalência e parâmetros do estimador assumidos para cada problema. Todas as bibliotecas recebem a mesma seed de dados em cada repetição e os mesmos limites principais de profundidade, tamanho mínimo de folha, árvores, amostragem de linhas e amostragem de features. Diferenças internas entre os algoritmos permanecem e são descritas nos resultados.

In [ ]:
SCENARIOS = {
    'credit_default': {'prevalence': 0.015, 'description': 'Inadimplência futura', 'max_depth': 8, 'min_leaf': 500, 'row_fraction': 0.80},
    'transaction_fraud': {'prevalence': 0.002, 'description': 'Fraude confirmada', 'max_depth': 9, 'min_leaf': 750, 'row_fraction': 0.80},
    'customer_churn': {'prevalence': 0.060, 'description': 'Encerramento futuro', 'max_depth': 7, 'min_leaf': 250, 'row_fraction': 0.80},
    'offer_acceptance': {'prevalence': 0.180, 'description': 'Aceitação de oferta', 'max_depth': 6, 'min_leaf': 300, 'row_fraction': 0.80},
}

def calibrate_intercept(signal, prevalence):
    signal = (signal - signal.mean()) / max(signal.std(), 1e-8)
    lo, hi = -30.0, 30.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if expit(signal + mid).mean() > prevalence:
            hi = mid
        else:
            lo = mid
    return signal + (lo + hi) / 2

def make_dataset(scenario, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(N_ROWS, N_FEATURES)).astype(np.float32)
    # Indicadores e variáveis monetárias com assimetria, comuns em dados bancários.
    X[:, 4:8] = rng.binomial(1, [0.12, 0.08, 0.18, 0.35], size=(N_ROWS, 4)).astype(np.float32)
    X[:, 8:12] = np.log1p(rng.lognormal(mean=2.0, sigma=0.8, size=(N_ROWS, 4))).astype(np.float32)
    X[:, 8:12] = (X[:, 8:12] - X[:, 8:12].mean(axis=0)) / (X[:, 8:12].std(axis=0) + 1e-6)
    # Proxies correlacionadas; todas continuam informativas para a verdade sintética.
    X[:, 16:20] = 0.75 * X[:, 0:4] + 0.66 * rng.normal(size=(N_ROWS, 4)).astype(np.float32)
    z = X

    if scenario == 'credit_default':
        signal = (1.0*z[:,0] + .85*z[:,1] + .65*z[:,2] + .5*z[:,3]
                  + .8*(z[:,4] > .5) + .7*(z[:,5] > .5) + .5*z[:,6] - .4*z[:,7]
                  + .65*z[:,8] - .55*z[:,9] + .45*z[:,10] + .35*z[:,11]
                  + .7*np.maximum(z[:,12]-.8, 0) + .6*(z[:,13]**2-1)
                  + .65*z[:,14]*z[:,15] + .18*z[:,16:24].sum(axis=1))
    elif scenario == 'transaction_fraud':
        signal = (1.0*z[:,0] + .8*np.maximum(z[:,1]-1, 0) + .7*(z[:,2] > 1.2)
                  + .9*z[:,3]*z[:,4] + .7*(z[:,5] > .5) + .5*z[:,6]
                  + .8*np.abs(z[:,7]) + .55*z[:,8] + .5*z[:,9]
                  + .65*(z[:,10]**2-1) + .7*z[:,11]*z[:,12]
                  + .5*z[:,13] + .4*z[:,14] + .35*z[:,15]
                  + .18*z[:,16:24].sum(axis=1))
    elif scenario == 'customer_churn':
        signal = (-.9*z[:,0] - .8*z[:,1] + .7*z[:,2] + .6*z[:,3]
                  + .65*(z[:,4] > .5) + .7*(z[:,5] > .5) + .5*z[:,6]
                  + .45*z[:,7] - .65*z[:,8] + .55*z[:,9] + .45*z[:,10]
                  + .6*np.maximum(z[:,11]-.7, 0) + .65*z[:,12]*z[:,13]
                  + .5*np.tanh(z[:,14]) + .4*z[:,15] + .18*z[:,16:24].sum(axis=1))
    elif scenario == 'offer_acceptance':
        signal = (.8*z[:,0] + .7*z[:,1] - .5*z[:,2] + .65*z[:,3]
                  + .6*(z[:,4] > .5) + .45*(z[:,5] > .5) + .5*z[:,6]
                  + .4*z[:,7] + .75*z[:,8] + .55*z[:,9] - .4*z[:,10]
                  + .6*(z[:,11]**2-1) + .65*z[:,12]*z[:,13]
                  + .5*np.tanh(z[:,14]) + .35*z[:,15] + .18*z[:,16:24].sum(axis=1))
    else:
        raise ValueError(f'Cenário desconhecido: {scenario}')
    logits = calibrate_intercept(signal, SCENARIOS[scenario]['prevalence'])
    y = rng.binomial(1, expit(logits)).astype(np.int8)
    return X, y, np.arange(N_FEATURES) < N_INFORMATIVE

print({name: info for name, info in SCENARIOS.items()})

## 7. Modelos e premissas dos hiperparâmetros

Os cinco braços usam importâncias de ganho e balanceamento de classe apropriado à API de cada biblioteca. Profundidade e tamanho mínimo de folha vêm das premissas do cenário; todos usam 100 árvores no modo completo, amostra de 80% das linhas e mtry próximo de √p. O Bankai usa aceleração por histograma com `max_bins=63`. LightGBM e XGBoost usam histogramas como modo eficiente próprio; essas diferenças de implementação são reportadas, não ocultadas. No cenário de fraude, `min_child_weight` do XGBoost usa a aproximação mais próxima para folhas ponderadas.

| Cenário | max_depth | min_samples_leaf | amostra de linhas |
|---|---:|---:|---:|
| Inadimplência | 8 | 500 | 80% |
| Fraude | 9 | 750 | 80% |
| Churn | 7 | 250 | 80% |
| Aceitação | 6 | 300 | 80% |

A folha mínima maior nos cenários com menos positivos reduz folhas sustentadas por poucos eventos. Todos os braços mantêm 100 árvores e feature sampling próximo de raiz de p.

In [ ]:
class XGBoostForestEstimator(XGBClassifier):
    # Expose one sklearn n_estimators value to Boruta while fitting that many
    # parallel trees in a single XGBoost boosting round.
    def fit(self, X, y, **kwargs):
        tree_count = self.n_estimators
        self.set_params(n_estimators=1, num_parallel_tree=tree_count)
        return super().fit(X, y, **kwargs)

class TrackedBoruta(BorutaPy):
    def _do_tests(self, dec_reg, hit_reg, iteration):
        state = super()._do_tests(dec_reg, hit_reg, iteration)
        self.decision_history_.append(state.copy())
        return state
    def fit(self, X, y):
        self.decision_history_ = []
        return super().fit(X, y)

def estimator_factory(model_name, seed, positive_weight, scenario):
    cfg = SCENARIOS[scenario]
    depth, leaf, row_fraction = cfg['max_depth'], cfg['min_leaf'], cfg['row_fraction']
    p_augmented = 2 * N_FEATURES
    node_fraction = min(1.0, math.sqrt(p_augmented) / p_augmented)
    common_seed = int(seed % (2**31 - 1))
    if model_name == 'BankaiRF (PyPI)':
        return BankaiRandomForestClassifier(
            n_estimators=N_TREES, max_depth=depth, min_samples_leaf=leaf, max_features='sqrt',
            max_samples=row_fraction, bootstrap=True, max_bins=MAX_BINS,
            importance_type='gain', class_weight='balanced',
            random_state=common_seed, n_jobs=N_JOBS)
    if model_name == 'Random Forest (sklearn)':
        return RandomForestClassifier(
            n_estimators=N_TREES, max_depth=depth, min_samples_leaf=leaf, max_features='sqrt',
            max_samples=row_fraction, bootstrap=True, class_weight='balanced',
            random_state=common_seed, n_jobs=N_JOBS)
    if model_name == 'Extra Trees':
        return ExtraTreesClassifier(
            n_estimators=N_TREES, max_depth=depth, min_samples_leaf=leaf, max_features='sqrt',
            bootstrap=True, max_samples=row_fraction, class_weight='balanced',
            random_state=common_seed, n_jobs=N_JOBS)
    if model_name == 'LightGBM RF':
        return LGBMClassifier(
            boosting_type='rf', objective='binary', n_estimators=N_TREES,
            max_depth=depth, num_leaves=min(2**depth, 256), min_child_samples=leaf, max_bin=MAX_BINS,
            bagging_fraction=row_fraction, bagging_freq=1,
            feature_fraction=1.0, feature_fraction_bynode=node_fraction,
            scale_pos_weight=positive_weight, importance_type='gain',
            random_state=common_seed, n_jobs=N_JOBS, verbosity=-1)
    if model_name == 'XGBoost RF':
        return XGBoostForestEstimator(
            objective='binary:logistic', n_estimators=1,
            num_parallel_tree=N_TREES, max_depth=depth, min_child_weight=leaf * 0.25, tree_method='hist',
            max_bin=MAX_BINS, subsample=row_fraction,
            colsample_bytree=1.0, colsample_bynode=node_fraction,
            learning_rate=1.0, scale_pos_weight=positive_weight,
            importance_type='gain', random_state=common_seed,
            n_jobs=N_JOBS, verbosity=0)
    raise ValueError(model_name)

MODEL_NAMES = ['BankaiRF (PyPI)', 'Random Forest (sklearn)', 'Extra Trees', 'LightGBM RF', 'XGBoost RF']

def make_selector(model_name, estimator_seed, selector_seed, positive_weight, scenario):
    estimator = estimator_factory(model_name, estimator_seed, positive_weight, scenario)
    return TrackedBoruta(estimator, n_estimators=N_TREES, max_iter=MAX_ROUNDS,
                         alpha=0.05, two_step=True, random_state=int(selector_seed % (2**31-1)),
                         verbose=0, early_stopping=False)

## 8. Aquecimento, medições e retomada

Cada repetição gera uma matriz e partição estratificada 80/20. Todos os cinco métodos recebem a mesma matriz de treino da mesma seed. O Boruta não vê o holdout. O aquecimento também usa uma partição 80/20 e fica fora das medições. A gravação incremental deixa retomar execuções interrompidas.

In [ ]:
def fit_partition(X, y, data_seed):
    train_idx, _ = train_test_split(np.arange(len(y)), test_size=0.20, stratify=y,
                                    random_state=int((data_seed + 73) % (2**31-1)))
    return X[train_idx], y[train_idx]

def target_round(selector, informative):
    for round_no, state in enumerate(selector.decision_history_, start=1):
        selected = state == 1
        tp = int(np.count_nonzero(selected & informative))
        fp = int(np.count_nonzero(selected & ~informative))
        recall = tp / int(informative.sum())
        fdr = fp / max(tp + fp, 1)
        if recall >= TARGET_RECALL and fdr <= MAX_FDR:
            return round_no
    return np.nan

def evaluate_one(X, y, informative, scenario, model_name, repeat_no, data_seed, model_seed, warmup=False):
    positive_weight = float((y == 0).sum() / max((y == 1).sum(), 1))
    selector_seed = model_seed + 900_001
    selector = make_selector(model_name, model_seed, selector_seed, positive_weight, scenario)
    start = time.perf_counter()
    with threadpool_limits(limits=N_JOBS):
        selector.fit(X, y)
    elapsed = time.perf_counter() - start
    selected = selector.support_
    tp = int(np.count_nonzero(selected & informative))
    fp = int(np.count_nonzero(selected & ~informative))
    recall = tp / int(informative.sum())
    fdr = fp / max(tp + fp, 1)
    return {
        'scenario': scenario, 'model': model_name, 'repeat': repeat_no,
        'data_seed': data_seed, 'model_seed': model_seed, 'boruta_seed': selector_seed,
        'warmup': warmup, 'runtime_seconds': elapsed,
        'rounds': len(selector.decision_history_), 'rounds_to_target': target_round(selector, informative),
        'confirmed_recall': recall, 'confirmed_fdr': fdr,
        'selected_count': int(selected.sum()), 'true_positive_count': tp,
        'false_positive_count': fp, 'tentative_count': int(selector.support_weak_.sum()),
        'positive_prevalence': float(y.mean()), 'rows': len(y), 'features': X.shape[1],
        'informative_features': int(informative.sum()), 'n_jobs': N_JOBS,
        'trees': N_TREES, 'max_depth': SCENARIOS[scenario]['max_depth'],
        'min_samples_leaf': SCENARIOS[scenario]['min_leaf'],
        'row_fraction': SCENARIOS[scenario]['row_fraction'], 'bankai_max_bins': MAX_BINS,
        'max_rounds': MAX_ROUNDS,
        'run_signature': RUN_SIGNATURE,
    }

def seed_for(scenario_index, repeat_index, model_index=0, purpose=0):
    return BASE_SEED + scenario_index*1_000_000 + repeat_index*10_000 + model_index*101 + purpose

RUN_SIGNATURE = json.dumps({'light_mode': LIGHT_MODE, 'rows': N_ROWS, 'features': N_FEATURES,
    'informative': N_INFORMATIVE, 'repeats': N_REPEATS, 'trees': N_TREES,
    'max_rounds': MAX_ROUNDS, 'base_seed': BASE_SEED, 'scenario_hyperparameters': SCENARIOS, 'n_jobs': N_JOBS,
    'max_bins': MAX_BINS, 'row_fraction': ROW_FRACTION, 'models': MODEL_NAMES}, sort_keys=True)
records = pd.read_csv(RAW_PATH).to_dict('records') if RAW_PATH.exists() else []
if records and any(r.get('run_signature') != RUN_SIGNATURE for r in records):
    raise RuntimeError('O arquivo de resultados usa outra configuração. Mova/remova o CSV ou use uma nova pasta.')
done = {(r['scenario'], r['model'], int(r['repeat'])) for r in records if not bool(r.get('warmup', False))}

def save_checkpoint(frame, path):
    tmp = path.with_name(path.name + '.tmp')
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

for scenario_index, scenario in enumerate(SCENARIOS):
    print()
    print(f'=== {scenario}: {SCENARIOS[scenario]["description"]} ===', flush=True)
    warm_data_seed = seed_for(scenario_index, 0, purpose=700)
    X_warm, y_warm, informative = make_dataset(scenario, warm_data_seed)
    X_warm, y_warm = fit_partition(X_warm, y_warm, warm_data_seed)
    for model_index, model_name in enumerate(MODEL_NAMES):
        warm_model_seed = seed_for(scenario_index, 0, model_index, purpose=800)
        print(f'warmup: {model_name}', flush=True)
        _ = evaluate_one(X_warm, y_warm, informative, scenario, model_name,
                         repeat_no=-1, data_seed=warm_data_seed,
                         model_seed=warm_model_seed, warmup=True)
        del _
        gc.collect()

    for repeat_no in range(N_REPEATS):
        missing = [name for name in MODEL_NAMES if (scenario, name, repeat_no) not in done]
        if not missing:
            continue
        data_seed = seed_for(scenario_index, repeat_no, purpose=500)
        X, y, informative = make_dataset(scenario, data_seed)
        X, y = fit_partition(X, y, data_seed)
        for model_index, model_name in enumerate(MODEL_NAMES):
            if model_name not in missing:
                continue
            model_seed = seed_for(scenario_index, repeat_no, model_index, purpose=900)
            print(f'{scenario} | repeat {repeat_no+1}/{N_REPEATS} | {model_name}', flush=True)
            row = evaluate_one(X, y, informative, scenario, model_name, repeat_no,
                               data_seed, model_seed)
            records.append(row)
            save_checkpoint(pd.DataFrame(records), RAW_PATH)
            done.add((scenario, model_name, repeat_no))
            print({k: row[k] for k in ('runtime_seconds','rounds','rounds_to_target','confirmed_recall','confirmed_fdr','selected_count')}, flush=True)
        del X, y
        gc.collect()

print()
print(f'Raw measurements: {RAW_PATH}')

## 9. Resumo dos resultados e melhor modelo por cenário

In [ ]:
raw = pd.read_csv(RAW_PATH)
raw = raw.loc[~raw['warmup'].astype(bool)].copy()
summary = (raw.groupby(['scenario','model'], sort=False)
    .agg(runs=('repeat','nunique'), max_depth=('max_depth','first'),
         min_samples_leaf=('min_samples_leaf','first'), row_fraction=('row_fraction','first'),
         bankai_max_bins=('bankai_max_bins','first'), n_jobs=('n_jobs','first'),
         runtime_median_s=('runtime_seconds','median'),
         runtime_iqr_s=('runtime_seconds', lambda s: s.quantile(.75)-s.quantile(.25)),
         recall_median=('confirmed_recall','median'), fdr_median=('confirmed_fdr','median'),
         selected_median=('selected_count','median'), tentative_median=('tentative_count','median'),
         rounds_median=('rounds','median'), target_round_median=('rounds_to_target','median'),
         target_success_rate=('rounds_to_target', lambda s: float(s.notna().mean())))
    .reset_index())
summary['quality_gate'] = ((summary['recall_median'] >= TARGET_RECALL) &
                           (summary['fdr_median'] <= MAX_FDR) &
                           (summary['target_success_rate'] >= .8))
summary['quality_rank'] = summary.groupby('scenario')['quality_gate'].transform(lambda s: s.astype(int))
summary = summary.sort_values(['scenario','quality_gate','recall_median','fdr_median',
                               'target_round_median','runtime_median_s'],
                              ascending=[True,False,False,True,True,True])
summary.to_csv(RESULT_DIR / ('summary-light.csv' if LIGHT_MODE else 'summary-full.csv'), index=False)
display(summary)

best = (summary.sort_values(['scenario','quality_gate','recall_median','fdr_median',
                             'target_round_median','runtime_median_s'],
                            ascending=[True,False,False,True,True,True])
        .drop_duplicates('scenario', keep='first'))
print('Melhor por cenário (critério: passa quality gate, recall, FDR, rodada-alvo e runtime):')
display(best[['scenario','model','quality_gate','recall_median','fdr_median',
              'target_round_median','runtime_median_s','runs']])
print('Resumo gravado em:', RESULT_DIR)

## 10. Como interpretar

O quality gate requer mediana de recall ≥ 0,95, mediana de FDR ≤ 0,05 e alvo alcançado em pelo menos 80% das repetições. Entre modelos que passam, a ordenação prioriza recall, FDR, rodada para atingir o alvo e runtime. Se nenhum passar, o notebook ainda mostra os resultados medidos e o melhor segundo esses critérios, mas não declara seleção confiável.

O vencedor pode mudar com prevalência, proporção de ruído, força dos sinais e recursos da máquina. A comparação é de Boruta usando a importância nativa de cada estimador; diferenças entre escalas/definições de importância fazem parte do comportamento prático do método. Para uso bancário real, valide em dados temporais separados, revise leakage, estabilidade por período e segmentos, e não use features selecionadas como evidência causal.

Parâmetros são escolhas iniciais justificadas pelo cenário, não uma busca de hiperparâmetros: crédito prioriza folhas maiores e profundidade moderada; fraude aceita mais profundidade, mas limita folhas; churn usa valores intermediários; aceitação usa árvores mais rasas. As quantidades de folhas representam aproximações de regularização em contagem de observações; para XGBoost, `min_child_weight` é massa Hessiana e não equivale exatamente a `min_samples_leaf`. Bankai usa aceleração por histograma com `max_bins=63`, conforme solicitado.